In [9]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [10]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

In [11]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.chat_models import init_chat_model

class EmailState(AgentState):
    email: str

model = init_chat_model(
    model_provider="openai",
    base_url=os.environ.get("OPENAI_BASE_URL"),
    api_key=os.environ.get("OPENAI_API_KEY"),
    model=os.environ.get("OPENAI_MODEL"),
    temperature=0.3,
)


agent = create_agent(
    model=model,
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [12]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

In [13]:
from rich import print as pprint

pprint(response)

{
    'messages': [
        HumanMessage(
            content='Please read my email and send a response immediately. Send the reply now in the same thread.',
            additional_kwargs={},
            response_metadata={},
            id='24a6ece2-7f43-4608-9ead-d4a283281c02'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 508,
                    'prompt_tokens': 85,
                    'total_tokens': 593,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 448,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0.00020745,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0.00020745,
                        'upstream_inference_prompt_cost': 4.25e-06,
                        'upstream_inference_completions_cost': 0.0002032
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-5-nano-2025-08-07',
                'system_fingerprint': None,
                'id': 'gen-1774600435-gEzhQ37xfdaEBXJVm1JF',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019d2e6d-b658-75a0-b526-cc93307dd06f-0',
            tool_calls=[
                {'name': 'read_email', 'args': {}, 'id': 'call_jVQhJmImYXbHrV8vmTF8iB7d', 'type': 'tool_call'}
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 85,
                'output_tokens': 508,
                'total_tokens': 593,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 448}
            }
        ),
        ToolMessage(
            content="Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John.",
            name='read_email',
            id='fb97a45a-6da1-4414-bc66-d15198fba976',
            tool_call_id='call_jVQhJmImYXbHrV8vmTF8iB7d'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 1363,
                    'prompt_tokens': 130,
                    'total_tokens': 1493,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 1216,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0.0005517,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0.0005517,
                        'upstream_inference_prompt_cost': 6.5e-06,
                        'upstream_inference_completions_cost': 0.0005452
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-5-nano-2025-08-07',
          

In [14]:
pprint(response['__interrupt__'])

[
    Interrupt(
        value={
            'action_requests': [
                {
                    'name': 'send_email',
                    'args': {
                        'body': "Hi John,\n\nNo problem—thanks for the heads up. I'm flexible tomorrow. What time 
works best for you? If it helps, I could do 9:30 AM, 1:00 PM, or 4:00 PM. If none of these fit, please suggest a 
time that suits you and I'll adjust.\n\nBest regards,\nSeán"
                    },
                    'description': 'Tool execution requires approval\n\nTool: send_email\nArgs: {\'body\': "Hi 
John,\\n\\nNo problem—thanks for the heads up. I\'m flexible tomorrow. What time works best for you? If it helps, I
could do 9:30 AM, 1:00 PM, or 4:00 PM. If none of these fit, please suggest a time that suits you and I\'ll 
adjust.\\n\\nBest regards,\\nSeán"}'
                }
            ],
            'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject']}]
        },
        id='42b6eb52c8f90b78d746908b1a8b891c'
    )
]

In [15]:
# Access just the 'body' argument from the tool call
pprint(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John,

No problem—thanks for the heads up. I'm flexible tomorrow. What time works best for you? If it helps, I could do 
9:30 AM, 1:00 PM, or 4:00 PM. If none of these fit, please suggest a time that suits you and I'll adjust.

Best regards,
Seán

## Approve

In [16]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

{
    'messages': [
        HumanMessage(
            content='Please read my email and send a response immediately. Send the reply now in the same thread.',
            additional_kwargs={},
            response_metadata={},
            id='24a6ece2-7f43-4608-9ead-d4a283281c02'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 508,
                    'prompt_tokens': 85,
                    'total_tokens': 593,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 448,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0.00020745,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0.00020745,
                        'upstream_inference_prompt_cost': 4.25e-06,
                        'upstream_inference_completions_cost': 0.0002032
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-5-nano-2025-08-07',
                'system_fingerprint': None,
                'id': 'gen-1774600435-gEzhQ37xfdaEBXJVm1JF',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019d2e6d-b658-75a0-b526-cc93307dd06f-0',
            tool_calls=[
                {'name': 'read_email', 'args': {}, 'id': 'call_jVQhJmImYXbHrV8vmTF8iB7d', 'type': 'tool_call'}
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 85,
                'output_tokens': 508,
                'total_tokens': 593,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 448}
            }
        ),
        ToolMessage(
            content="Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John.",
            name='read_email',
            id='fb97a45a-6da1-4414-bc66-d15198fba976',
            tool_call_id='call_jVQhJmImYXbHrV8vmTF8iB7d'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 1363,
                    'prompt_tokens': 130,
                    'total_tokens': 1493,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 1216,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0.0005517,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0.0005517,
                        'upstream_inference_prompt_cost': 6.5e-06,
                        'upstream_inference_completions_cost': 0.0005452
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-5-nano-2025-08-07',
          

## Reject

In [18]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{
    'messages': [
        HumanMessage(
            content='Please read my email and send a response immediately. Send the reply now in the same thread.',
            additional_kwargs={},
            response_metadata={},
            id='24a6ece2-7f43-4608-9ead-d4a283281c02'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 508,
                    'prompt_tokens': 85,
                    'total_tokens': 593,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 448,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0.00020745,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0.00020745,
                        'upstream_inference_prompt_cost': 4.25e-06,
                        'upstream_inference_completions_cost': 0.0002032
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-5-nano-2025-08-07',
                'system_fingerprint': None,
                'id': 'gen-1774600435-gEzhQ37xfdaEBXJVm1JF',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019d2e6d-b658-75a0-b526-cc93307dd06f-0',
            tool_calls=[
                {'name': 'read_email', 'args': {}, 'id': 'call_jVQhJmImYXbHrV8vmTF8iB7d', 'type': 'tool_call'}
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 85,
                'output_tokens': 508,
                'total_tokens': 593,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 448}
            }
        ),
        ToolMessage(
            content="Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John.",
            name='read_email',
            id='fb97a45a-6da1-4414-bc66-d15198fba976',
            tool_call_id='call_jVQhJmImYXbHrV8vmTF8iB7d'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 1363,
                    'prompt_tokens': 130,
                    'total_tokens': 1493,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 1216,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0.0005517,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0.0005517,
                        'upstream_inference_prompt_cost': 6.5e-06,
                        'upstream_inference_completions_cost': 0.0005452
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-5-nano-2025-08-07',
          

In [17]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

KeyError: '__interrupt__'

## Edit

In [18]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{
    'messages': [
        HumanMessage(
            content='Please read my email and send a response immediately. Send the reply now in the same thread.',
            additional_kwargs={},
            response_metadata={},
            id='87825db3-e432-49d3-bd92-aae32f1c154d'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 1585,
                    'prompt_tokens': 85,
                    'total_tokens': 1670,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 1536,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0.00063825,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0.00063825,
                        'upstream_inference_prompt_cost': 4.25e-06,
                        'upstream_inference_completions_cost': 0.000634
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-5-nano-2025-08-07',
                'system_fingerprint': None,
                'id': 'gen-1774599199-gtYk5IFLAG2jGDs0lj8C',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019d2e5a-d6df-70e2-a1f8-203b90da974e-0',
            tool_calls=[
                {'name': 'read_email', 'args': {}, 'id': 'call_rbNVGao00zU2ZZJ3rT9ceEr9', 'type': 'tool_call'}
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 85,
                'output_tokens': 1585,
                'total_tokens': 1670,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 1536}
            }
        ),
        ToolMessage(
            content="Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John.",
            name='read_email',
            id='a8d84208-ce9a-40e5-bed9-f24689290b6a',
            tool_call_id='call_rbNVGao00zU2ZZJ3rT9ceEr9'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 1387,
                    'prompt_tokens': 130,
                    'total_tokens': 1517,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 1280,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0.0005613,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0.0005613,
                        'upstream_inference_prompt_cost': 6.5e-06,
                        'upstream_inference_completions_cost': 0.0005548
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-5-nano-2025-08-07',
     